In [14]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "utils").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print("Using project root:", ROOT)

import mne
import pandas as pd
import numpy as np
from typing import Callable
from IPython.display import clear_output

from utils.ica_pipeline import preprocess_for_ica, detrend_and_iir_bandpass
from utils.epoch_and_average import revise_annot

Using project root: /Users/jowanglin/regression-based_ERP


In [3]:
WEIHUN_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/weihun"
JY_DIR = "/Users/jowanglin/Word-Position-Effect_BLP-lab/data/eeg/jy"
OUT_FOLDER = "preprocessed"

### Preprocessing
- Reset annotations
    - `"w{n}/condition"`, `n` starts from 1 to the second-to-last word of each trial
- Add channel montage
- Re-reference
- Filter
- Save as FIF on disk

In [4]:
def reref_and_filt(raw: mne.io.Raw, *,
                   fixation: str,
                   non_final: str,
                   codes_after_fixation_are_final: bool,
                   transform_description: Callable,
                   ref_channels: dict,
                   l_freq: float=0.1, h_freq: float=30.0,
                   order: int=2,
                   ftype: str="butter",
                   **conditions) -> mne.io.Raw:
    raw = raw.copy()
    annot = raw.annotations
    df_annot = pd.DataFrame(annot)

    annot_revised = revise_annot(df_annot,
                                 fixation=fixation,
                                 non_final=non_final,
                                 codes_after_fixation_are_final=codes_after_fixation_are_final,
                                 transform_description=transform_description,
                                 **conditions)
    raw = raw.set_annotations(annot_revised)

    raw.load_data()
    raw_reref = preprocess_for_ica(raw.copy(),
                                   ref_channels=ref_channels,
                                   return_raw_reref_only=True)
    raw_reref_filt = detrend_and_iir_bandpass(raw_reref,
                                              l_freq=l_freq, h_freq=h_freq,
                                              order=order,
                                              ftype=ftype)
    return raw_reref_filt


In [5]:
fixation = "210"
non_final = "222"
pos = ("231", "232")
neg = ("233", "234")
neu = ("235", "236", "237", "238")

num = 3
raw_file_name = f"subj{str(num).zfill(3)}_EML_YA_run01-12.set"
raw = mne.io.read_raw_eeglab(f"{WEIHUN_DIR}/{raw_file_name}", verbose=False, preload=False)

raw_reref_filt = reref_and_filt(raw,
                                fixation=fixation,
                                non_final=non_final,
                                codes_after_fixation_are_final=True,
                                transform_description=lambda arr: [a.replace("boundary", "edge") for a in arr],
                                ref_channels={"M2": 0.5},
                                l_freq=0.1, h_freq=30.0,
                                order=2,
                                ftype="butter",
                                pos=pos, neu=neu, neg=neg)

df_annot = pd.DataFrame(raw_reref_filt.annotations)
display(df_annot.head(15))
print(f"\nTotal number of events = {df_annot.shape[0]}")   # should be 1566 (from run01-06) + 1385 (from run07-12) + 1 (EEGLAB should insert 'boundary' at event 1567)
display(df_annot.iloc[1560: 1570])                         # check event 1567 (marked as 'boundary' by EEGLAB) is converted to 'edge' in mne.Annotations
                                                           # because raw.filter() by default skip_by_annotation=('edge', 'bad_acq_skip')
print(raw_reref_filt.info)



Reading 0 ... 2457439  =      0.000 ...  2457.439 secs...


/var/folders/n3/nqqvqq1n0jx25dfd3nyxjxmh0000gn/T/ipykernel_50680/3419391356.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(f"{WEIHUN_DIR}/{raw_file_name}", verbose=False, preload=False)


standard_montage is not provided; defaulting to MNE-shipped standard_1020 and renaming all channel names to upper case.
Creating RawArray with float64 data, n_channels=33, n_times=2457440
    Range : 0 ... 2457439 =      0.000 ...  2457.439 secs
Ready.


,onset,duration,description,orig_time,extras
0,7.275,0.0,210,None,{}
1,8.291,0.0,w1/neu,None,{}
2,8.658,0.0,w2/neu,None,{}
3,9.024,0.0,w3/neu,None,{}
4,9.391,0.0,w4/neu,None,{}
5,9.757,0.0,w5/neu,None,{}
6,10.124,0.0,w6/neu,None,{}
7,10.491,0.0,w7/neu,None,{}
8,10.857,0.0,235,None,{}
9,11.057,0.0,1,None,{}



Total number of events = 2952


,onset,duration,description,orig_time,extras
1560,1292.3570,0.0,241,None,{}
1561,1293.8730,0.0,250,None,{}
1562,1294.9540,0.0,251,None,{}
1563,1296.4550,0.0,237,None,{}
1564,1298.7870,0.0,2,None,{}
1565,1298.7970,0.0,3,None,{}
1566,1300.5195,0.0,edge,None,{}
1567,1308.8370,0.0,210,None,{}
1568,1309.8530,0.0,w1/neu,None,{}
1569,1310.2200,0.0,w2/neu,None,{}


<Info | 8 non-empty values
 bads: []
 ch_names: FP1, FP2, F7, F3, FZ, F4, F8, FT7, FC3, FCZ, FC4, FT8, T7, C3, ...
 chs: 31 EEG, 2 EOG
 custom_ref_applied: False
 dig: 34 items (3 Cardinal, 31 EEG)
 highpass: 0.1 Hz
 lowpass: 30.0 Hz
 meas_date: unspecified
 nchan: 33
 projs: []
 sfreq: 1000.0 Hz
>


In [104]:
fixation = "210"
non_final = "222"
pos = ("231", "232")
neg = ("233", "234")
neu = ("235", "236", "237", "238")

subj_ids, preprocessed = [], []
for num in range(1, 40):
    raw_file_name = f"subj{str(num).zfill(3)}_EML_YA_run01-12.set"
    try:
        raw = mne.io.read_raw_eeglab(f"{WEIHUN_DIR}/{raw_file_name}", verbose=False, preload=False)
    except FileNotFoundError:
        continue
    raw_reref_filt = reref_and_filt(raw,
                                    fixation=fixation,
                                    non_final=non_final,
                                    codes_after_fixation_are_final=True,
                                    transform_description=lambda arr: [a.replace("boundary", "edge").replace("-99", "edge")
                                                                       for a in arr],
                                    ref_channels={"M2": 0.5},
                                    l_freq=0.1, h_freq=30.0,
                                    order=2,
                                    ftype="butter",
                                    pos=pos, neu=neu, neg=neg)
    raw_reref_filt.save(f"{WEIHUN_DIR}/{OUT_FOLDER}/subj{str(num).zfill(3)}_raw.fif", overwrite=True)
    subj_ids.append(raw_file_name)
    preprocessed.append(raw_reref_filt)
    clear_output()
    

### Checking each raw file's event code for missing trials

In [110]:
template = np.sort([str(num) for num in list(range(1, 209))])
subj_desc = [pd.DataFrame(r.annotations)["description"].to_numpy() for r in preprocessed]
subj_indices = [np.array([i for i, x in enumerate(desc) if x.startswith("w") and "/" in x]) for desc in subj_desc]
subj_diffs = [np.append(np.diff(indices), 0) for indices in subj_indices]
subj_trial_codes = [desc[indices[diffs != 1]+2]
                       for desc, indices, diffs in zip(subj_desc, subj_indices, subj_diffs)]
subj_trial_codes = [[t for t in trial_codes] for trial_codes in subj_trial_codes]
#subj_numeric_check = np.where(np.array([all([t.isnumeric() for t in trial_codes])
                                                  #for trial_codes in subj_trial_codes]) != 1)[0]
subj_trial_codes = np.array([[str(t) for t in trial_codes]
                                     for trial_codes in subj_trial_codes], dtype=object)
mask = np.arange(len(subj_trial_codes))
#good_subj_trial_codes = [sorted(trial_codes) for trial_codes in subj_trial_codes[~np.isin(mask, subj_numeric_check)]]
subj_trial_codes_sorted = [np.sort(trial_codes) for trial_codes in subj_trial_codes]
subj_trials_check = np.where(np.array([np.array_equal(trial_codes, template)
                                         for trial_codes in subj_trial_codes_sorted]) != 1)[0]
print(f"Subjects with missing/problematic trials:\n  Indices: {subj_trials_check}\n  File names: {np.array(subj_ids)[subj_trials_check]}")


Subjects with missing/problematic trials:
  Indices: [ 0 26]
  File names: ['subj002_EML_YA_run01-12.set' 'subj029_EML_YA_run01-12.set']


In [139]:
print(f"{subj_ids[subj_trials_check[0]].split('_')[0]} missing trials (unique item codes):")
print("  ", sorted([str(c) for c in (set(template) - set(subj_trial_codes[0]))]))

print(f"\n{subj_ids[subj_trials_check[0]].split('_')[0]} corrupted trial:")
display(pd.DataFrame(preprocessed[26].annotations).head(15))


subj002 missing trials (unique item codes):
   ['141', '142', '143', '144']

subj002 corrupted trial:


,onset,duration,description,orig_time,extras
0,6.331,0.0,210,None,{}
1,7.347,0.0,w1,None,{}
2,7.713,0.0,w2/neu,None,{}
3,8.080,0.0,w3/neu,None,{}
4,8.446,0.0,146/neu,None,{}
5,8.813,0.0,w4/neu,None,{}
6,9.180,0.0,w5/neu,None,{}
7,9.546,0.0,w6/neu,None,{}
8,9.913,0.0,237,None,{}
9,10.113,0.0,208,None,{}
